# EDA — Base completa da escola (Censo Escolar, 455 colunas)

Como ficou confirmado no `03_eda_escola.ipynb` (Seção 4.3), o JOIN entre `alunos` e `escola` por `id_escola` não é viável — são dois sistemas de identificação diferentes, e isso está fora do meu controle (depende da fonte original dos dados, não é algo que eu resolvo editando código). Então mudei o foco: em vez de tentar enriquecer os alunos com dados de escola, vou explorar a base `escola_completo` (a ingestão nova, com as 455 colunas reais confirmadas, path `bronze/br_inep_censo_escolar/escola_completo/`) de forma independente, sem depender de nenhum JOIN.

O objetivo deste notebook é responder duas perguntas:

1. Das 455 colunas, quantas realmente têm preenchimento suficiente pra virar feature?
2. Existe, dentro da própria base de escola, alguma coluna que sirva como variável-alvo — ou seja, algum indicador de resultado/desempenho educacional (e não só característica de entrada, como infraestrutura ou corpo técnico)? Isso é crítico porque, sem o JOIN com `alunos`, não tenho mais acesso à taxa de alfabetização calculada a partir da prova (a que eu vinha usando desde o Dia 1) — preciso saber se existe um substituto nativo da própria base do Censo Escolar.

Esse notebook não parte de nenhuma suposição sobre o que vou encontrar — a ideia é ler a base inteira e deixar os dados falarem, do mesmo jeito que fiz no `03_eda_escola.ipynb` pro problema do `id_escola`.

In [ ]:
import sys
sys.path.append("../..")

import boto3
import pandas as pd
import numpy as np
from pathlib import Path

from src.preprocessing.load_data import BUCKET, _ler_parquet_do_prefixo

s3 = boto3.client("s3")

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 100)

## 1. Carga da base `escola_completo` (com cache local)

Uso o mesmo padrão de cache que já uso em `load_data.py` pra `base_modelagem.parquet`: na primeira vez lê do S3 (path confirmado no `03_eda_escola.ipynb`, Seção 7) e salva em `data/processed/escola_completo.parquet`; das próximas vezes lê direto do disco local, sem precisar baixar do S3 de novo. Se eu já tiver rodado a Seção 7.1 do notebook anterior antes, esse cache já existe e essa célula só lê do disco (rápido).

In [ ]:
PREFIXO_BRONZE_ESCOLA_COMPLETO = "bronze/br_inep_censo_escolar/escola_completo/"
CACHE_ESCOLA_COMPLETO = Path("../..") / "data" / "processed" / "escola_completo.parquet"

if CACHE_ESCOLA_COMPLETO.exists():
    print(f"Lendo do cache local: {CACHE_ESCOLA_COMPLETO}")
    escola_completo = pd.read_parquet(CACHE_ESCOLA_COMPLETO)
else:
    print(f"Cache local não encontrado, lendo do S3: s3://{BUCKET}/{PREFIXO_BRONZE_ESCOLA_COMPLETO}")
    escola_completo = _ler_parquet_do_prefixo(BUCKET, PREFIXO_BRONZE_ESCOLA_COMPLETO)
    CACHE_ESCOLA_COMPLETO.parent.mkdir(parents=True, exist_ok=True)
    escola_completo.to_parquet(CACHE_ESCOLA_COMPLETO, index=False)
    print(f"Cache salvo em: {CACHE_ESCOLA_COMPLETO}")

print(f"\nescola_completo: {escola_completo.shape[0]:,} linhas, {escola_completo.shape[1]} colunas")
print(f"Uso de memória: {escola_completo.memory_usage(deep=True).sum() / 1e6:,.1f} MB")

## 2. Preenchimento por coluna (% de nulos)

Antes de qualquer análise mais profunda, preciso saber quais das 455 colunas realmente têm dado. Monto uma tabela-resumo com tipo, quantidade preenchida, quantidade nula, % de nulo e quantidade de valores únicos por coluna, ordenada da menos preenchida pra mais preenchida.

In [ ]:
resumo_colunas = pd.DataFrame({
    "coluna": escola_completo.columns,
    "dtype": [str(escola_completo[c].dtype) for c in escola_completo.columns],
    "qtd_preenchido": [escola_completo[c].notna().sum() for c in escola_completo.columns],
    "qtd_nulo": [escola_completo[c].isna().sum() for c in escola_completo.columns],
    "valores_unicos": [escola_completo[c].nunique(dropna=True) for c in escola_completo.columns],
})
resumo_colunas["pct_nulo"] = (resumo_colunas["qtd_nulo"] / len(escola_completo) * 100).round(2)
resumo_colunas = resumo_colunas.sort_values("pct_nulo", ascending=False).reset_index(drop=True)

print(f"Total de colunas: {len(resumo_colunas)}")
print(f"Colunas 100% nulas: {(resumo_colunas['pct_nulo'] == 100).sum()}")
print(f"Colunas com mais de 50% de nulo: {(resumo_colunas['pct_nulo'] > 50).sum()}")
print(f"Colunas com menos de 5% de nulo: {(resumo_colunas['pct_nulo'] < 5).sum()}")

resumo_colunas.head(30)

Exporto a tabela completa (455 linhas) em CSV pra eu poder olhar com calma e colar aqui se precisar de ajuda pra interpretar o padrão de preenchimento.

In [ ]:
print(resumo_colunas.to_csv(index=False))

## 3. Lista completa dos nomes de coluna

O catálogo (`catalogo-dados-fase3.md`) organiza as colunas por categoria (identificação, infraestrutura, equipamentos, corpo técnico, matrículas por perfil demográfico etc.), mas foi montado olhando o schema de forma manual — não é garantido que cobre as 455 colunas uma por uma. Antes de procurar por uma variável-alvo, quero ver a lista crua e completa, sem filtro nenhum.

In [ ]:
print(f"Total: {len(escola_completo.columns)} colunas\n")
for i, coluna in enumerate(sorted(escola_completo.columns), start=1):
    print(f"{i:3d}. {coluna}")

## 4. Busca por candidatos a variável-alvo

Sem o JOIN com `alunos`, a única forma de ter uma variável-alvo é encontrar algo dentro da própria `escola_completo` que meça resultado/desempenho educacional — não característica de entrada (infraestrutura, corpo técnico, matrícula), mas sim um indicador de saída (aprovação, reprovação, rendimento, proficiência, abandono etc.).

O catálogo já apontou uma coluna com relação direta ao tema, `programa_brasil_alfabetizado` — mas essa é uma flag de participação em programa, não necessariamente um indicador de resultado. Preciso verificar se existe algo mais parecido com uma taxa/índice de desempenho, procurando por palavra-chave em todos os 455 nomes de coluna (não só nas categorias que já vieram documentadas no catálogo, que pode não ser exaustivo).

In [ ]:
palavras_chave_desempenho = [
    "aprova", "reprov", "rendimento", "ideb", "proficien", "desempenho",
    "indicador", "distorcao", "abandono", "conclu", "evasao", "nota",
    "resultado", "sucesso", "fracasso", "permanenc", "aprendiz",
    "alfabet", "taxa", "avaliacao", "saeb", "prova",
]

candidatos = {}
for palavra in palavras_chave_desempenho:
    encontrados = [c for c in escola_completo.columns if palavra in c.lower()]
    if encontrados:
        candidatos[palavra] = encontrados

if not candidatos:
    print("Nenhuma coluna bateu com nenhuma das palavras-chave de desempenho/resultado.")
else:
    for palavra, colunas in candidatos.items():
        print(f"'{palavra}': {colunas}")

### 4.1 Conferindo `programa_brasil_alfabetizado` separadamente

Mesmo não sendo um indicador de resultado, vale olhar o preenchimento e a distribuição dessa coluna — é a única que o catálogo já tinha identificado com relação direta ao tema.

In [ ]:
if "programa_brasil_alfabetizado" in escola_completo.columns:
    coluna_pba = escola_completo["programa_brasil_alfabetizado"]
    print(f"dtype: {coluna_pba.dtype}")
    print(f"% preenchido: {coluna_pba.notna().mean() * 100:.2f}%")
    print("\nDistribuição de valores:")
    print(coluna_pba.value_counts(dropna=False))
else:
    print("Coluna 'programa_brasil_alfabetizado' não encontrada em escola_completo.")

## 5. Olhando de perto cada candidato encontrado na Seção 4

Pra cada coluna que apareceu na busca por palavra-chave, reviso preenchimento e distribuição — se for numérica, uso `.describe()`; se for categórica/baixa cardinalidade, uso `.value_counts()`. Isso ajuda a separar candidato real (indicador de resultado, com dado de verdade) de falso positivo (coluna que só tem a palavra no nome mas é outra coisa — por exemplo, `taxa` pode aparecer em nome de coluna de infraestrutura sem ser desempenho educacional).

In [ ]:
todas_colunas_candidatas = sorted(set(coluna for cols in candidatos.values() for coluna in cols))

for coluna in todas_colunas_candidatas:
    print("=" * 70)
    print(f"Coluna: {coluna}")
    print(f"dtype: {escola_completo[coluna].dtype}")
    print(f"% preenchido: {escola_completo[coluna].notna().mean() * 100:.2f}%")
    print(f"valores únicos: {escola_completo[coluna].nunique(dropna=True)}")
    if pd.api.types.is_numeric_dtype(escola_completo[coluna]):
        print("\n.describe():")
        print(escola_completo[coluna].describe())
    else:
        print("\n.value_counts() (top 15):")
        print(escola_completo[coluna].value_counts(dropna=False).head(15))
    print()

## 6. Conclusão — existe uma variável-alvo viável dentro da `escola_completo`?

Rodando as Seções 2, 4 e 5, o resultado foi claro: nenhuma das colunas que bateram na busca por palavra-chave (Seção 4) é de fato um indicador de resultado/desempenho educacional a nível de escola. A grande maioria são falsos positivos — nomes que contêm palavras como "taxa" ou "indicador" mas descrevem infraestrutura, corpo técnico ou matrícula, não desempenho de fato. O único candidato genuinamente ligado ao tema, `programa_brasil_alfabetizado`, é uma flag de participação em programa (sim/não), não uma medida de resultado — não dá pra usar como alvo.

**Conclusão: não existe uma variável-alvo nativa viável dentro da `escola_completo`.** As 455 colunas cobrem muito bem características de entrada (infraestrutura, corpo docente/técnico, matrícula, localização, gestão), mas nenhuma mede resultado educacional de saída.

Isso é uma decisão de arquitetura do projeto — o que vira a variável-alvo define o resto da modelagem. Diante desse resultado, decidi buscar uma fonte externa de indicador de desempenho por escola em vez de insistir num alvo nativo que não existe, ou voltar pro grão de aluno (o que perderia a riqueza das 455 colunas de infraestrutura). Essa busca é o que me levou ao dataset `br_inep_ideb` no `06_eda_ideb_escola.ipynb` — o IDEB é justamente um indicador de resultado por escola, calculado pelo próprio INEP, e passou a ser a base da variável-alvo definitiva do projeto a partir dali.